# 🤟 Arabic Sign Language Recognition — Exploration & Baseline

Exploratory notebook for the capstone project **Sign-Link Innovators (TrioDigi)**.
It covers dataset loading (KArSL-502), video inspection, and a **baseline** CNN+LSTM
(EfficientNetB0 + LSTM) trained on a 100-class subset.

> The final model reported in the project (MobileNetV2 + BiLSTM, 50 signs) and the Streamlit app
> are described in the project README.

**Set the dataset location first:** `export KARSL_DIR=/path/to/KArSL-502` (or edit `DATA_DIR` below).
Expected layout: `general_train/<sign_id>/*.mp4`, `general_test/<sign_id>/*.mp4`, `KARSL-502_Labels.xlsx`.

## 1. Setup

In [ ]:
import os, re, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from arabic_reshaper import reshape
from bidi.algorithm import get_display
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

DATA_DIR  = Path(os.environ.get("KARSL_DIR", "./data"))
TRAIN_DIR = DATA_DIR / "general_train"
TEST_DIR  = DATA_DIR / "general_test"
print("GPUs:", tf.config.list_physical_devices("GPU"))
for n, p in [("DATA_DIR", DATA_DIR), ("TRAIN_DIR", TRAIN_DIR), ("TEST_DIR", TEST_DIR)]:
    print(f"{n}: exists={p.exists()}")

## 2. Load labels

In [ ]:
# ================================================
# Section 2: Load Labels
# ================================================

labels_path = DATA_DIR / "KARSL-502_Labels.xlsx"

if labels_path.exists():
    df_labels = pd.read_excel(labels_path)

    label_dict = dict(zip(df_labels['SignID'].astype(str), df_labels['Sign-Arabic'].astype(str)))

    label_names = df_labels['Sign-Arabic'].astype(str).tolist()
    NUM_CLASSES = len(label_names)

    print(f" Labels loaded: {NUM_CLASSES} إشارة")
    print(f"الأعمدة: {list(df_labels.columns)}")
    print(f"مثال: SignID '1' = '{label_dict.get('1')}'")
    display(df_labels[['SignID', 'Sign-Arabic', 'Sign-English']].head(10))
else:
    print("❌ ملف الـ Labels غير موجود!")

## 3. Collect videos (train / test)

In [ ]:
def collect_videos(root: Path, split: str) -> pd.DataFrame:
    rows = []
    for vid in root.rglob("*"):
        if vid.suffix.lower() not in {".mp4", ".mov", ".avi", ".mkv"}:
            continue
        nums    = re.findall(r"\d+", vid.stem.split("_(")[0])
        sign_id = str(int(nums[-1])) if nums else "0"
        rows.append({
            "path"    : str(vid),
            "sign_id" : sign_id,
            "label"   : label_dict.get(sign_id, "Unknown"),
            "split"   : split,
            "filename": vid.name,
        })
    return pd.DataFrame(rows).sort_values("sign_id").reset_index(drop=True)

df_train = collect_videos(TRAIN_DIR, "train")
df_test  = collect_videos(TEST_DIR,  "test")

print(f"✅ Train: {len(df_train)} فيديو")
print(f"✅ Test : {len(df_test)} فيديو")
print(f"⚠️  Unknown — Train: {(df_train.label=='Unknown').sum()} | Test: {(df_test.label=='Unknown').sum()}")
display(df_train[["filename","sign_id","label"]].head(8))

## 4. Inspect a sample video

In [ ]:
sample = df_train.iloc[0]
cap = cv2.VideoCapture(sample["path"])
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()
print("Total frames:", len(frames))

idx = np.linspace(0, len(frames) - 1, 6, dtype=int)
title = get_display(reshape(f"الإشارة: {sample['label']}  |  Sign ID: {sample['sign_id']}"))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, i in zip(axes.flatten(), idx):
    ax.imshow(frames[i]); ax.set_title(f"Frame {i}"); ax.axis("off")
plt.suptitle(title, fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Data generator
Samples `frames_per_video` frames per video, resizes and normalises them.

In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.utils import Sequence

# ========================== Settings ==========================
base_path = str(TRAIN_DIR)
frames_per_video = 16
img_size = (112, 112)        # small input to fit GPU memory
batch_size = 4
max_videos_per_class = None      
# ============================================================

class_folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]
class_folders.sort()
# keep only folders that are plain numeric sign IDs (skips odd folders such as '0001-0070')
selected_classes = [c for c in class_folders if c.isdigit()][:100]

print(f"📊 عدد الكلاسات المكتشفة: {len(selected_classes)}")
print(f"أول 10 كلاسات: {selected_classes[:10]}")

# ====================== Data generator ======================
class SignLanguageGenerator(Sequence):
    def __init__(self, base_path, selected_classes, frames_per_video=16, img_size=(128,128), 
                 batch_size=8, max_videos_per_class=None, shuffle=True):
        self.base_path = base_path
        self.selected_classes = selected_classes
        self.frames_per_video = frames_per_video
        self.img_size = img_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        
        self.video_list = []
        self.labels = []
        
        for cls_idx, cls in enumerate(selected_classes):
            cls_path = os.path.join(base_path, cls)
            videos = [v for v in os.listdir(cls_path) if v.endswith(('.mp4', '.MP4'))]
            if max_videos_per_class:
                videos = videos[:max_videos_per_class]
            
            for v in videos:
                self.video_list.append(os.path.join(cls_path, v))
                self.labels.append(cls_idx)
        
        self.video_list = np.array(self.video_list)
        self.labels = np.array(self.labels)
        self.on_epoch_end()
        
        print(f"✅ إجمالي فيديوهات للتدريب: {len(self.video_list)}")
    
    def __len__(self):
        return int(np.ceil(len(self.video_list) / self.batch_size))
    
    def on_epoch_end(self):
        if self.shuffle:
            idx = np.random.permutation(len(self.video_list))
            self.video_list = self.video_list[idx]
            self.labels = self.labels[idx]
    
    def __getitem__(self, index):
        batch_videos = self.video_list[index*self.batch_size:(index+1)*self.batch_size]
        batch_labels = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        X_batch = []
        for vid_path in batch_videos:
            cap = cv2.VideoCapture(vid_path)
            frames = []
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            step = max(1, total_frames // self.frames_per_video)
            
            for i in range(self.frames_per_video):
                cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.resize(frame, self.img_size)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame / 255.0
                frames.append(frame)
            cap.release()
            
            # pad short videos
            while len(frames) < self.frames_per_video:
                frames.append(frames[-1] if frames else np.zeros((*self.img_size, 3)))
            
            X_batch.append(frames)
        
        return np.array(X_batch, dtype=np.float32), np.array(batch_labels, dtype=np.int32)


train_generator = SignLanguageGenerator(
    base_path=base_path,
    selected_classes=selected_classes,
    frames_per_video=frames_per_video,
    img_size=img_size,
    batch_size=batch_size,
    max_videos_per_class=max_videos_per_class
)

## 6. Baseline model — EfficientNetB0 (frozen) + 2×LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

# Transfer learning: frozen EfficientNetB0 per frame -> LSTM over time
base_model = EfficientNetB0(
    weights='imagenet', 
    include_top=False, 
    input_shape=(img_size[0], img_size[1], 3)
)
base_model.trainable = False   # Freezes the CNN first

model = models.Sequential([
    layers.TimeDistributed(base_model, input_shape=(frames_per_video, img_size[0], img_size[1], 3)),
    layers.TimeDistributed(layers.GlobalAveragePooling2D()),
    
    layers.LSTM(512, return_sequences=True, dropout=0.3),
    layers.LSTM(256, dropout=0.4),
    
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dense(len(selected_classes), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Train
No validation split is used in this baseline, so callbacks monitor the training metrics.

In [ ]:
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy("mixed_float16")   # saves GPU memory

train_generator = SignLanguageGenerator(
    base_path=base_path, selected_classes=selected_classes,
    frames_per_video=frames_per_video, img_size=img_size,
    batch_size=batch_size, max_videos_per_class=None, shuffle=True)

callbacks = [
    EarlyStopping(monitor="accuracy", patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="accuracy", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]
history = model.fit(train_generator, epochs=20, callbacks=callbacks, verbose=1)